# Model Training and evaluation For Linear Regression

In [1]:
import pandas as pd

df = pd.read_csv("code/household_power_consumption.txt", sep=";", low_memory=False, na_values="?")

In [2]:
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [3]:
import pandas as pd

# Display the first few rows of the DataFrame
print(df.head())
# Display the shape of the DataFrame
print("Shape of the DataFrame:", df.shape)
# Display the column names
print("Column names:", df.columns.tolist())
# Display the data types of each column
print("Data types of each column:")

         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  
Shape of the DataFrame: (2075259, 9)
Column names: ['Date', 'Time', 'Global_active_power', 'Global_

In [3]:
import pandas as pd

# Strip column names of extra spaces
df.columns = df.columns.str.strip()

# Combine 'Date' and 'Time' into a new 'DateTime' column
df["DateTime"] = pd.to_datetime(df["Date"] + ' ' + df["Time"], format="%d/%m/%Y %H:%M:%S", errors='coerce')

# Drop the old Date and Time columns if not needed
df.drop(columns=["Date", "Time"], inplace=True)

# Convert numeric columns from object to float
cols_to_convert = df.columns.difference(["DateTime"])  # Exclude DateTime from conversion
for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Show the updated data types
print(df.dtypes)

Global_active_power             float64
Global_reactive_power           float64
Voltage                         float64
Global_intensity                float64
Sub_metering_1                  float64
Sub_metering_2                  float64
Sub_metering_3                  float64
DateTime                 datetime64[ns]
dtype: object


In [6]:
df.columns

Index(['Global_active_power', 'Global_reactive_power', 'Voltage',
       'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3', 'DateTime'],
      dtype='object')

# Linear Regression

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# --- Step 1: Define Features and Target ---
# Drop rows with NaN (if not already done)
df.dropna(inplace=True)

# Target variable
y = df['Global_active_power']

# Features (drop target and DateTime)
X = df[['Global_reactive_power', 'Voltage','Global_intensity', 'Sub_metering_1', 'Sub_metering_2','Sub_metering_3']]

# --- Step 2: Split into Train & Test ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Step 3: Train Linear Regression ---
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred = lr_pipeline.predict(X_test)

# --- Step 4: Evaluate the Model ---
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Linear Regression Results:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")


Linear Regression Results:
Mean Squared Error: 0.0016
R² Score: 0.9986


In [8]:

# Preprocessing and feature selection
# Model creation for predicting bmi using preg,bp,st
x = df[['Global_reactive_power', 'Voltage','Global_intensity', 'Sub_metering_1', 'Sub_metering_2','Sub_metering_3']] #features
y = df['Global_active_power'] #target
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=50) # Test_size = 20% for testing and 80% for training

# Model training
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train,y_train) # training xtrain and ytrain
print(model.coef_)
print(model.intercept_)


[-0.17563155  0.00444147  0.23802985 -0.00033072 -0.00043807  0.00217741]
-1.070997697608627


In [9]:
# Model evaluation using testing split
from sklearn import metrics
import numpy as np
y_pred=model.predict(x_test)
print('Mean Absolute Error:', metrics.mean_absolute_error(y_test, y_pred))
print('Mean Squared Error:', metrics.mean_squared_error(y_test, y_pred))
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))
from sklearn.metrics import r2_score
r2_score(y_test, y_pred)

Mean Absolute Error: 0.025981078432041886
Mean Squared Error: 0.0016672172962529238
Root Mean Squared Error: 0.040831572297095345


0.9985106492715757

In [10]:
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

def tune_linear_models():
    models = {
        "Linear": LinearRegression(),
        "Ridge_0.1": Ridge(alpha=0.1),
        "Ridge_1": Ridge(alpha=1),
        "Lasso_0.1": Lasso(alpha=0.1),
        "Lasso_1": Lasso(alpha=1),
    }

    best_mse = float('inf')
    best_model = None
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        mse = mean_squared_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        print(f"{name} → MSE: {mse:.4f}, R²: {r2:.4f}")
        if mse < best_mse:
            best_mse = mse
            best_model = model
    return best_model


Got accuracy in linear regression arround 0.998

# Random Forest

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# --- 1. Prepare Features & Target ---
x = df[['Global_reactive_power', 'Voltage','Global_intensity', 'Sub_metering_1', 'Sub_metering_2','Sub_metering_3']] #features
y = df['Global_active_power'] 

# --- 2. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 3. Simple Random Forest Model ---
rf = RandomForestRegressor(n_estimators=5,max_depth=10,random_state=42,n_jobs=-1)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# --- 4. Evaluation ---
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("✅ Fast Random Forest:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")


✅ Fast Random Forest:
Mean Squared Error: 0.0012
R² Score: 0.9989


In [18]:
from sklearn.ensemble import RandomForestRegressor

def tune_random_forest():
    best_mse = float('inf')
    best_model = None

    for n in [25, 50]:
        for d in [None, 10]:
            rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=42, n_jobs=-1)
            rf.fit(X_train, y_train)
            preds = rf.predict(X_test)
            mse = mean_squared_error(y_test, preds)
            r2 = r2_score(y_test, preds)
            print(f"RF (n={n}, depth={d}) → MSE: {mse:.4f}, R²: {r2:.4f}")
            if mse < best_mse:
                best_mse = mse
                best_model = rf
    return best_model


### Gradient Boosting

In [13]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# --- 1. Prepare Features & Target ---
x = df[['Global_reactive_power', 'Voltage','Global_intensity', 'Sub_metering_1', 'Sub_metering_2','Sub_metering_3']] #features
y = df['Global_active_power'] 

# --- 2. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 3. Gradient Boosting Regressor (Lightweight Settings) ---
gbr = GradientBoostingRegressor(n_estimators=5, learning_rate=0.1, max_depth=3, random_state=42)

gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)

# --- 4. Evaluation ---
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("🚀 Gradient Boosting Results:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")


🚀 Gradient Boosting Results:
Mean Squared Error: 0.4067
R² Score: 0.6382


In [14]:
from sklearn.ensemble import GradientBoostingRegressor

def tune_gb():
    best_mse = float('inf')
    best_model = None

    for lr in [0.1, 0.05]:
        for n in [50, 100]:
            gb = GradientBoostingRegressor(n_estimators=n, learning_rate=lr, random_state=42)
            gb.fit(X_train, y_train)
            preds = gb.predict(X_test)
            mse = mean_squared_error(y_test, preds)
            r2 = r2_score(y_test, preds)
            print(f"GB (n={n}, lr={lr}) → MSE: {mse:.4f}, R²: {r2:.4f}")
            if mse < best_mse:
                best_mse = mse
                best_model = gb
    return best_model


### Neural Networks

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# --- 1. Prepare Features & Target ---
x = df[['Global_reactive_power', 'Voltage','Global_intensity', 'Sub_metering_1', 'Sub_metering_2','Sub_metering_3']] #features
y = df['Global_active_power'] 

# --- 2. Split and Scale ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 3. Convert to Tensors ---
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# --- 4. Create DataLoader ---
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# --- 5. Define Model ---
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(X_train_tensor.shape[1], 64)
        self.fc2 = nn.Linear(64, 32)
        self.out = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.out(x)

model = Net()

# --- 6. Training Setup ---
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# --- 7. Train ---
for epoch in range(20):  # small number of epochs to keep it fast
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    if epoch % 5 == 0:
        print(f"Epoch {epoch} - Loss: {loss.item():.4f}")

# --- 8. Predict & Evaluate ---
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor).numpy()

mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("\n🧠 PyTorch Neural Net Results:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")


# Best Model Saved

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ========== DATA LOADING & PREPROCESSING ==========
def load_and_preprocess():
    df = pd.read_csv("code/household_power_consumption.txt", 
                    sep=";", 
                    low_memory=False, 
                    na_values="?")
    
    # Clean and transform data
    df.columns = df.columns.str.strip()
    df["DateTime"] = pd.to_datetime(df["Date"] + ' ' + df["Time"], 
                                   format="%d/%m/%Y %H:%M:%S", 
                                   errors='coerce')
    df.drop(columns=["Date", "Time"], inplace=True)
    df.dropna(inplace=True)
    
    # Convert numeric columns
    numeric_cols = df.columns.difference(["DateTime"])
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    return df

# ========== MODEL TRAINING & EVALUATION ==========
def train_evaluate_models(X_train, X_test, y_train, y_test):
    results = {}
    
    # Linear Models
    linear_models = {
        'Linear': LinearRegression(),
        'Ridge': Ridge(alpha=0.5),
        'Lasso': Lasso(alpha=0.1)
    }
    
    for name, model in linear_models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        results[name] = {
            'mse': mean_squared_error(y_test, preds),
            'r2': r2_score(y_test, preds)
        }
    
    # Random Forest (optimized settings)
    rf = RandomForestRegressor(n_estimators=50, 
                             max_depth=10, 
                             random_state=42, 
                             n_jobs=-1)
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)
    results['RandomForest'] = {
        'mse': mean_squared_error(y_test, preds),
        'r2': r2_score(y_test, preds)
    }
    
    # Gradient Boosting (optimized settings)
    gb = GradientBoostingRegressor(n_estimators=100, 
                                 learning_rate=0.1, 
                                 max_depth=3, 
                                 random_state=42)
    gb.fit(X_train, y_train)
    preds = gb.predict(X_test)
    results['GradientBoosting'] = {
        'mse': mean_squared_error(y_test, preds),
        'r2': r2_score(y_test, preds)
    }
    
    return results

# ========== LIGHTWEIGHT NEURAL NET ==========
class PowerConsumptionNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    
    def forward(self, x):
        return self.net(x)

def train_nn(X_train, X_test, y_train, y_test, epochs=15, batch_size=64):
    # Standardize data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Convert to tensors
    X_train_t = torch.FloatTensor(X_train_scaled)
    X_test_t = torch.FloatTensor(X_test_scaled)
    y_train_t = torch.FloatTensor(y_train.values).view(-1, 1)
    y_test_t = torch.FloatTensor(y_test.values).view(-1, 1)
    
    # Create model
    model = PowerConsumptionNN(X_train.shape[1])
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Training loop
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    for epoch in range(epochs):
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
    
    # Evaluation
    model.eval()
    with torch.no_grad():
        predictions = model(X_test_t).numpy()
    
    return {
        'mse': mean_squared_error(y_test, predictions),
        'r2': r2_score(y_test, predictions)
    }

# ========== MAIN EXECUTION ==========
if __name__ == "__main__":
    # Load and prepare data
    df = load_and_preprocess()
    X = df[['Global_reactive_power', 'Voltage', 'Global_intensity', 
            'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']]
    y = df['Global_active_power']
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train and evaluate models
    results = train_evaluate_models(X_train, X_test, y_train, y_test)
    
    # Train and evaluate NN
    nn_results = train_nn(X_train, X_test, y_train, y_test)
    results['NeuralNetwork'] = nn_results
    
    # Print results
    print("\n=== Model Performance ===")
    for model, metrics in results.items():
        print(f"{model}:")
        print(f"  MSE: {metrics['mse']:.4f}")
        print(f"  R²: {metrics['r2']:.4f}\n")
    
    # Save best model (Random Forest in most cases)
    best_model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
    best_model.fit(X_train, y_train)
    joblib.dump(best_model, "power_consumption_model.pkl")
    print("Best model saved as 'power_consumption_model.pkl'")


=== Model Performance ===
Linear:
  MSE: 0.0016
  R²: 0.9986

Ridge:
  MSE: 0.0016
  R²: 0.9986

Lasso:
  MSE: 0.0027
  R²: 0.9976

RandomForest:
  MSE: 0.0012
  R²: 0.9989

GradientBoosting:
  MSE: 0.0012
  R²: 0.9990

NeuralNetwork:
  MSE: 0.0010
  R²: 0.9991

Best model saved as 'power_consumption_model.pkl'
